In [1]:
import pandas as pd

# --- Load Data ---
orders = pd.read_csv("C:/Users/AKSHIT/Downloads/retail-orders-raw.csv")
dictionary = pd.read_csv("C:/Users/AKSHIT/Downloads/retail-data-dictionary.csv")

# --- Completeness Checks ---
required_cols = ["order_id","order_date","city","category","quantity","unit_price","payment_status"]
completeness = orders[required_cols].notnull().mean() * 100
print("Completeness % by column:\n", completeness)

# --- Uniqueness Check ---
duplicates = orders.duplicated(subset=["order_id"]).sum()
print("Duplicate order_id count:", duplicates)

# --- Validity Checks ---
# Date validity
orders["order_date_parsed"] = pd.to_datetime(orders["order_date"], errors="coerce")
invalid_dates = orders["order_date_parsed"].isna().sum()
print("Invalid date count:", invalid_dates)

# Quantity validity
invalid_qty = (orders["quantity"].apply(pd.to_numeric, errors="coerce") <= 0).sum()
print("Invalid quantity count:", invalid_qty)

# Discount validity
orders["discount_pct_num"] = pd.to_numeric(orders["discount_pct"], errors="coerce")
invalid_discount = ((orders["discount_pct_num"] < 0) | (orders["discount_pct_num"] > 100)).sum()
print("Invalid discount count:", invalid_discount)

# --- Consistency Checks ---
# Normalize categories
orders["customer_segment_norm"] = orders["customer_segment"].str.strip().str.title()
orders["payment_status_norm"] = orders["payment_status"].str.strip().str.title()
print("Unique normalized segments:", orders["customer_segment_norm"].unique())
print("Unique normalized payment statuses:", orders["payment_status_norm"].unique())

# --- Freshness Check ---
latest_date = orders["order_date_parsed"].max()
print("Latest order date:", latest_date)

# --- KPI Calculations ---
orders["net_revenue"] = (
    pd.to_numeric(orders["quantity"], errors="coerce") *
    orders["unit_price"] *
    (1 - orders["discount_pct_num"].fillna(0)/100)
)

total_orders = orders["order_id"].nunique()
revenue = orders.loc[orders["payment_status_norm"]=="Paid","net_revenue"].sum()
aov = revenue / total_orders

print("Total Orders:", total_orders)
print("Revenue (Paid only):", revenue)
print("Average Order Value:", aov)


Completeness % by column:
 order_id          100.000000
order_date         91.666667
city               91.666667
category          100.000000
quantity          100.000000
unit_price        100.000000
payment_status    100.000000
dtype: float64
Duplicate order_id count: 1
Invalid date count: 3
Invalid quantity count: 1
Invalid discount count: 1
Unique normalized segments: ['Student' 'Fresher' 'Professional']
Unique normalized payment statuses: ['Paid' 'Pending' 'Failed' 'Refunded']
Latest order date: 2026-01-18 00:00:00
Total Orders: 11
Revenue (Paid only): 8470.5
Average Order Value: 770.0454545454545


In [2]:
# --- Threshold Enforcement ---
def check_threshold(name, value, threshold, comparator=">="):
    if comparator == ">=" and value >= threshold:
        print(f"{name}: PASS ({value:.2f}%)")
    elif comparator == "<=" and value <= threshold:
        print(f"{name}: PASS ({value:.2f}%)")
    else:
        print(f"{name}: FAIL ({value:.2f}%)")

# Completeness
for col, pct in completeness.items():
    check_threshold(f"Completeness {col}", pct, 98)

# Uniqueness
dup_rate = duplicates / total_orders * 100
check_threshold("Duplicate Rate", dup_rate, 0.5, "<=")

# Validity
valid_rate = (len(orders) - (invalid_dates + invalid_qty + invalid_discount)) / len(orders) * 100
check_threshold("Validity", valid_rate, 99)

# Freshness
days_since_latest = (pd.Timestamp.today() - latest_date).days
if days_since_latest <= 7:
    print("Freshness: PASS (latest order within 7 days)")
else:
    print(f"Freshness: FAIL (latest order {days_since_latest} days old)")


Completeness order_id: PASS (100.00%)
Completeness order_date: FAIL (91.67%)
Completeness city: FAIL (91.67%)
Completeness category: PASS (100.00%)
Completeness quantity: PASS (100.00%)
Completeness unit_price: PASS (100.00%)
Completeness payment_status: PASS (100.00%)
Duplicate Rate: FAIL (9.09%)
Validity: FAIL (58.33%)
Freshness: FAIL (latest order 218 days old)


In [3]:
# --- Escalation Log ---
escalations = []

def log_issue(dimension, result, owner, action):
    escalations.append({"Dimension": dimension, "Result": result, "Owner": owner, "Action Required": action})

# Completeness
if completeness["order_date"] < 98 or completeness["city"] < 98:
    log_issue("Completeness", "FAIL", "Data Steward", "Investigate missing values, enforce mandatory entry")

# Uniqueness
if dup_rate > 0.5:
    log_issue("Uniqueness", "FAIL", "Ops Manager", "Deduplicate and strengthen order ID generation")

# Validity
if valid_rate < 99:
    log_issue("Validity", "FAIL", "Data Quality Lead", "Correct invalid dates/quantities/discounts")

# Freshness
if days_since_latest > 7:
    log_issue("Freshness", "FAIL", "Ops Manager", "Check ingestion schedule, refresh pipeline")

# Save escalation log
pd.DataFrame(escalations).to_csv("escalation_log.csv", index=False)
print("Escalation log written to escalation_log.csv")


Escalation log written to escalation_log.csv


In [4]:
# --- KPI Summary ---
kpi_summary = {
    "Total Orders": total_orders,
    "Revenue (Paid only)": revenue,
    "Average Order Value": aov,
    "Latest Order Date": str(latest_date.date())
}

# Merge KPI + Escalations
summary_df = pd.DataFrame([kpi_summary])
pd.DataFrame(escalations).to_csv("escalation_log.csv", index=False)
summary_df.to_csv("kpi_summary.csv", index=False)

print("KPI summary written to kpi_summary.csv")


KPI summary written to kpi_summary.csv


In [5]:
# View escalation log
esc_log = pd.read_csv("escalation_log.csv")
print(esc_log)

# View KPI summary
kpi_log = pd.read_csv("kpi_summary.csv")
print(kpi_log)


      Dimension Result              Owner  \
0  Completeness   FAIL       Data Steward   
1    Uniqueness   FAIL        Ops Manager   
2      Validity   FAIL  Data Quality Lead   
3     Freshness   FAIL        Ops Manager   

                                     Action Required  
0  Investigate missing values, enforce mandatory ...  
1     Deduplicate and strengthen order ID generation  
2         Correct invalid dates/quantities/discounts  
3         Check ingestion schedule, refresh pipeline  
   Total Orders  Revenue (Paid only)  Average Order Value Latest Order Date
0            11               8470.5           770.045455        2026-01-18


In [1]:
import pandas as pd

print(pd.read_csv("kpi_summary.csv"))
print(pd.read_csv("escalation_log.csv"))


   Total Orders  Revenue (Paid only)  Average Order Value Latest Order Date
0            11               8470.5           770.045455        2026-01-18
      Dimension Result              Owner  \
0  Completeness   FAIL       Data Steward   
1    Uniqueness   FAIL        Ops Manager   
2      Validity   FAIL  Data Quality Lead   
3     Freshness   FAIL        Ops Manager   

                                     Action Required  
0  Investigate missing values, enforce mandatory ...  
1     Deduplicate and strengthen order ID generation  
2         Correct invalid dates/quantities/discounts  
3         Check ingestion schedule, refresh pipeline  
